In [1]:
import os
from pathlib import Path
import shutil

TEXBIN = "/Library/TeX/texbin"
if Path(TEXBIN).is_dir():
    os.environ["PATH"] = TEXBIN + ":" + os.environ.get("PATH", "")

print("pdflatex now:", shutil.which("pdflatex"))
print("latexmk now:", shutil.which("latexmk"))

pdflatex now: /Library/TeX/texbin/pdflatex
latexmk now: /Library/TeX/texbin/latexmk


In [2]:
# main.py
from pathlib import Path
import numpy as np
from openai import OpenAI
import secrets

from config import get_experiment
from data_generation import generate_timeseries_dataset, stats_json
from variable_format import build_var_map_from_df_cols, remove_variables_from_first_paragraph, apply_variable_math
from llm_calls import llm_extended_body, llm_table_caption, llm_title
from tables import df_to_latex_summary_table, df_to_latex_full_table, insert_after_first_paragraph
from latex_build import build_tex_doc, compile_tex, show_first_latex_error
from notation_families import choose_family_sequential, apply_notation_to_dataframe, save_mapping_json


def main():
    experiment = get_experiment()
    client = OpenAI()

    base_seed = experiment.get("rng_seed", None)
    if base_seed is None:
        base_seed = secrets.randbits(32)  # different each script run
    rng_base = np.random.default_rng(base_seed)

    OUT = Path(experiment["out_dir"])
    OUT.mkdir(parents=True, exist_ok=True)

    cue_words = [
        "rapid early uptake",
        "gradual approach to equilibrium",
        "time-dependent adsorption",
        "kinetic uptake behavior",
        "run-to-run kinetic variation",
        "early-stage adsorption",
        "rate-limited uptake",
        "equilibrium approach",
        "phosphate uptake kinetics",
        "pseudo-first-order adsorption"
    ]

    for i in range(1, experiment["NUM_DATASETS"] + 1):
        print(f"▶ Dataset {i}/{experiment['NUM_DATASETS']}")

        paper_id = f"PFO_paper{i:02d}"

        # Per-report seed (so each report gets a different dataset)
        if experiment.get("per_report_seed", True):
            report_seed = int(rng_base.integers(0, 2**32 - 1))
            rng_report = np.random.default_rng(report_seed)
        else:
            report_seed = None
            rng_report = rng_base

        # Cue word for title variation
        title_cue = cue_words[int(rng_report.integers(0, len(cue_words)))]

        # Generate dataset
        df = generate_timeseries_dataset(experiment, rng_report)

        # Choose notation family sequentially and rename columns
        notation = choose_family_sequential(experiment, i)
        df, col_rename_map = apply_notation_to_dataframe(df, notation)

        # Save mapping + seed
        save_mapping_json(OUT, paper_id, notation, col_rename_map, report_seed)

        stats = stats_json(df)

        # Build variable map from actual dataframe columns (excluding run_id)
        cols_for_varmap = [c for c in df.columns if c != "run_id"]
        VARMAP = build_var_map_from_df_cols(cols_for_varmap)
        VARS = sorted(VARMAP.keys(), key=len, reverse=True)

        # LLM write-up
        body_raw = llm_extended_body(client, experiment, stats)

        # Remove variable names in first paragraph
        body_raw = remove_variables_from_first_paragraph(body_raw, VARS)

        # Convert variable mentions to math italics
        body = apply_variable_math(body_raw, VARMAP, VARS)

        # ---- Table 1: summary (insert once, after first paragraph) ----
        cap1 = llm_table_caption(client, experiment, 1, df, table_kind="Summary statistics")
        table1_tex = df_to_latex_summary_table(df, cap1, label="tab:summary", VARMAP=VARMAP)
        body = insert_after_first_paragraph(body, table1_tex)

        # ---- Table 2: full dataset (append once, at end) ----
        if experiment.get("INCLUDE_FULL_TABLE", False):
            cap2 = llm_table_caption(client, experiment, 2, df, table_kind="Full recorded dataset")
            table2_tex = df_to_latex_full_table(df, cap2, label="tab:full", VARMAP=VARMAP, max_rows=None)
            body = body + "\n\n" + table2_tex

        # Title (make sure your llm_title accepts title_cue)
        title = llm_title(client, experiment, body, stats, title_cue)

        tex_doc = build_tex_doc(title, body)
        assert tex_doc.count("{") == tex_doc.count("}"), "Brace mismatch before pdflatex"

        tex_out = OUT / f"{paper_id}.tex"
        tex_out.write_text(tex_doc, encoding="utf-8")

        ok, log = compile_tex(tex_out, OUT)
        if not ok:
            (OUT / f"{paper_id}.log.txt").write_text(log, encoding="utf-8")
            print(f"   ✖ compile failed; saved log to {paper_id}.log.txt")
            print("\n--- First LaTeX error snippet ---")
            print(show_first_latex_error(log))
            raise RuntimeError("pdflatex failed; inspect the snippet above and the saved log.")

        pdf_path = OUT / f"{paper_id}.pdf"
        if not pdf_path.exists():
            raise RuntimeError("Compilation succeeded but PDF not found where expected.")

        # Remove intermediate files (keep only PDF)
        for ext in [".aux", ".log", ".out", ".tex"]:
            f = OUT / f"{paper_id}{ext}"
            if f.exists():
                f.unlink()

        print("   ✔ saved", pdf_path)

    print("🎉 Done. Files saved in:", OUT.resolve())


if __name__ == "__main__":
    main()

▶ Dataset 1/4
   ✔ saved Output/PFO_paper01.pdf
▶ Dataset 2/4
   ✔ saved Output/PFO_paper02.pdf
▶ Dataset 3/4
   ✔ saved Output/PFO_paper03.pdf
▶ Dataset 4/4
   ✔ saved Output/PFO_paper04.pdf
🎉 Done. Files saved in: /Users/md_halim_mondol/Desktop/STEPS-LLM/steps-llm/Src/Single Equation/Pseudo-first-order/Output
